# 90 - Validate Medallion

Runs structural and reconciliation checks after the Silver and Gold notebooks. Results are saved to `metadata.data_quality_results`.

In [ ]:
# Attach the schema-enabled Lakehouse containing the Bronze shortcuts and
# make it the default Lakehouse before running this notebook.
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
METADATA_SCHEMA = "metadata"
COMPANY_KEY = "NAC"  # Change when processing another GP company database.

def q(identifier: str) -> str:
    return f"`{identifier.replace('`', '``')}`"

def sql_string(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"

def table_exists(schema_name: str, table_name: str) -> bool:
    try:
        return spark.catalog.tableExists(f"{schema_name}.{table_name}")
    except Exception:
        return False

def require_tables(table_names: list[str], schema_name: str = BRONZE_SCHEMA) -> None:
    missing = [name for name in table_names if not table_exists(schema_name, name)]
    if missing:
        raise RuntimeError(
            f"Missing tables in {schema_name}: {', '.join(missing)}. "
            "Confirm the OneLake shortcuts and exact table names."
        )

company = sql_string(COMPANY_KEY)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {q(SILVER_SCHEMA)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {q(GOLD_SCHEMA)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {q(METADATA_SCHEMA)}")
print(f"Company: {COMPANY_KEY}; Bronze schema: {BRONZE_SCHEMA}")

In [ ]:
from datetime import datetime, timezone
from pyspark.sql.types import StructType, StructField, StringType, LongType

required_gold = [
    "dim_customer", "dim_location", "dim_employee", "dim_technician",
    "dim_contract", "dim_equipment", "dim_date", "dim_job",
    "fact_service_call", "fact_service_appointment", "fact_service_cost",
    "fact_service_labor", "fact_contract_schedule", "fact_contract_revenue",
    "fact_job_financial_snapshot", "fact_job_cost", "fact_job_billing"
]
require_tables(required_gold, GOLD_SCHEMA)

## Execute checks

In [ ]:
checks = [
    ("gold_customer_key_unique", "ERROR", f"SELECT COUNT(*) - COUNT(DISTINCT customer_key) FROM {GOLD_SCHEMA}.dim_customer"),
    ("gold_location_key_unique", "ERROR", f"SELECT COUNT(*) - COUNT(DISTINCT location_key) FROM {GOLD_SCHEMA}.dim_location"),
    ("gold_employee_key_unique", "ERROR", f"SELECT COUNT(*) - COUNT(DISTINCT employee_key) FROM {GOLD_SCHEMA}.dim_employee"),
    ("gold_technician_key_unique", "ERROR", f"SELECT COUNT(*) - COUNT(DISTINCT technician_key) FROM {GOLD_SCHEMA}.dim_technician"),
    ("gold_contract_key_unique", "ERROR", f"SELECT COUNT(*) - COUNT(DISTINCT contract_key) FROM {GOLD_SCHEMA}.dim_contract"),
    ("gold_job_key_unique", "ERROR", f"SELECT COUNT(*) - COUNT(DISTINCT job_key) FROM {GOLD_SCHEMA}.dim_job"),
    ("gold_job_snapshot_key_unique", "ERROR", f"SELECT COUNT(*) - COUNT(DISTINCT job_snapshot_key) FROM {GOLD_SCHEMA}.fact_job_financial_snapshot"),
    ("bronze_silver_change_order_amount", "ERROR", f"""
        WITH source_totals AS (
            SELECT TRIM(WS_Job_Number) AS job_number,
                   CAST(SUM(CAST(change_order_est_cost AS DECIMAL(18,2))) AS DECIMAL(18,2)) AS amount
            FROM {q(BRONZE_SCHEMA)}.JC01001
            WHERE TRIM(WS_Job_Number) <> ''
            GROUP BY TRIM(WS_Job_Number)
        )
        SELECT COUNT(*)
        FROM {q(SILVER_SCHEMA)}.job_financial_snapshot s
        LEFT JOIN source_totals b ON s.job_number = b.job_number
        WHERE s.company_key = {company}
          AND NOT (s.change_order_est_cost <=> COALESCE(b.amount, CAST(0 AS DECIMAL(18,2))))
    """),
    ("silver_gold_change_order_amount", "ERROR", f"""
        SELECT COUNT(*)
        FROM {q(SILVER_SCHEMA)}.job_financial_snapshot s
        FULL OUTER JOIN {q(GOLD_SCHEMA)}.fact_job_financial_snapshot g
          ON s.company_key = g.company_key AND s.job_number = g.job_number
         AND s.snapshot_date = g.snapshot_date
        WHERE s.job_number IS NULL OR g.job_number IS NULL
           OR NOT (s.change_order_est_cost <=> g.change_order_est_cost)
    """),
    ("change_orders_without_snapshot_job", "WARN", f"""
        SELECT COUNT(DISTINCT TRIM(b.WS_Job_Number))
        FROM {q(BRONZE_SCHEMA)}.JC01001 b
        LEFT JOIN {q(SILVER_SCHEMA)}.job_financial_snapshot s
          ON s.company_key = {company} AND s.job_number = TRIM(b.WS_Job_Number)
        WHERE TRIM(b.WS_Job_Number) <> '' AND s.job_number IS NULL
    """),
    ("service_call_key_unique", "ERROR", f"SELECT COUNT(*) - COUNT(DISTINCT service_call_key) FROM {GOLD_SCHEMA}.fact_service_call"),
    ("silver_gold_service_cost_count", "ERROR", f"SELECT ABS((SELECT COUNT(*) FROM {SILVER_SCHEMA}.service_cost) - (SELECT COUNT(*) FROM {GOLD_SCHEMA}.fact_service_cost))"),
    ("silver_gold_service_cost_amount", "ERROR", f"SELECT CAST(ABS((SELECT COALESCE(SUM(document_amount),0) FROM {SILVER_SCHEMA}.service_cost) - (SELECT COALESCE(SUM(document_amount),0) FROM {GOLD_SCHEMA}.fact_service_cost)) AS BIGINT)"),
    ("silver_gold_job_cost_count", "ERROR", f"SELECT ABS((SELECT COUNT(*) FROM {SILVER_SCHEMA}.job_cost) - (SELECT COUNT(*) FROM {GOLD_SCHEMA}.fact_job_cost))"),
    ("silver_gold_job_cost_amount", "ERROR", f"SELECT CAST(ABS((SELECT COALESCE(SUM(document_amount),0) FROM {SILVER_SCHEMA}.job_cost) - (SELECT COALESCE(SUM(document_amount),0) FROM {GOLD_SCHEMA}.fact_job_cost)) AS BIGINT)"),
    ("unknown_service_customers", "WARN", f"SELECT COUNT(*) FROM {GOLD_SCHEMA}.fact_service_call WHERE customer_key = 0"),
    ("unknown_service_locations", "WARN", f"SELECT COUNT(*) FROM {GOLD_SCHEMA}.fact_service_call WHERE location_key = 0"),
    ("unknown_service_technicians", "WARN", f"SELECT COUNT(*) FROM {GOLD_SCHEMA}.fact_service_call WHERE technician_key = 0"),
    ("unknown_job_cost_jobs", "WARN", f"SELECT COUNT(*) FROM {GOLD_SCHEMA}.fact_job_cost WHERE job_key = 0"),
    ("unknown_job_billing_jobs", "WARN", f"SELECT COUNT(*) FROM {GOLD_SCHEMA}.fact_job_billing WHERE job_key = 0"),
]

run_id = datetime.now(timezone.utc).isoformat()
results = []
for check_name, severity, statement in checks:
    try:
        value = spark.sql(statement).first()[0]
        numeric_value = int(value or 0)
        status = "PASS" if numeric_value == 0 else ("FAIL" if severity == "ERROR" else "WARN")
        error_message = None
    except Exception as exc:
        numeric_value = None
        status = "ERROR"
        error_message = f"{type(exc).__name__}: {str(exc)[:1000]}"
    results.append((run_id, check_name, severity, status, numeric_value, error_message))

result_schema = StructType([
    StructField("run_id_utc", StringType(), False),
    StructField("check_name", StringType(), False),
    StructField("severity", StringType(), False),
    StructField("status", StringType(), False),
    StructField("observed_value", LongType(), True),
    StructField("error_message", StringType(), True),
])
result_df = spark.createDataFrame(results, result_schema)
result_df.write.format("delta").mode("append").saveAsTable(f"{METADATA_SCHEMA}.data_quality_results")
display(result_df.orderBy("severity", "check_name"))

## Fail the pipeline on structural or reconciliation errors

In [ ]:
failed = [r.check_name for r in result_df.filter("status IN ('FAIL', 'ERROR') AND severity = 'ERROR'").collect()]
if failed:
    raise RuntimeError("Medallion validation failed: " + ", ".join(failed))
print("All ERROR-level checks passed. Review WARN rows before publishing the semantic model.")